# 전체90 archive 자동 조사 READY — CPU / Colab
고정 source/input 두 작은 ZIP만 업로드합니다. Cache/raw archive 업로드·전체 다운로드·GPU가 필요하지 않습니다.
Run all 한 번으로 source 검증·의존성·인증·mount·runtime metadata pin·계획 검증·CPU worker 시작을 진행합니다. 기본 ACTION=prepare_and_start입니다. 매segment 승인은 없습니다.
전체90각2scan 측정 allocation입니다. **전체 인덱싱/무결성 완료가 아니며 전체 실행으로 자동 전환하지 않습니다.** train/test RGB·pose·depth를 scene별로 조사합니다. Pair 수는 별도 대조 전까지 미확정입니다.
총 시간 상한은 없습니다. 실제 경과 timer와 status를 보고 오래 걸리면 stop을 요청하세요. 요청별 network timeout5초, 고정 source/HTTP/IO/storage/RAM/disk cap은 유지합니다. Colab 자체 세션 종료를 방지하거나 자동 재시작하지 않습니다.
첫 pin/plan은 Drive prepared 폴더에 readback 저장합니다. 재개는 같은 input/source ZIP·prepared·store를 그대로 사용하며 repin/새budget를 만들지 않습니다. 기존 실패 calibration store는 건드리지 않습니다.

In [ ]:
from pathlib import Path
import os,sys,time,json,uuid,hashlib,zipfile,subprocess,shutil
sys.dont_write_bytecode = True
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
ACTION = 'prepare_and_start'  # prepare_and_start / prepare / status / stop / resume / recover
SOURCE_REVISION = '__ROOT_REVISION__'
SOURCE_ZIP_NAME = 'rewind_fleet_survey_source_ROOT.zip'
INPUT_ZIP_NAME = 'fleet_survey_inputs_ROOT.zip'
SOURCE_ZIP_SHA256 = '__ROOT_SOURCE_SHA256__'
INPUT_ZIP_SHA256 = '__ROOT_INPUT_SHA256__'
EXPECTED_REQUEST_SHA256 = 'db0f21d90e605d20c959c080c48e1cd7e8cb858958ab846c738f4f7b86c9fb4e'
LOCAL_ROOT = Path('/content/fleet_survey_runtime')
SOURCE_DIR = LOCAL_ROOT / 'source'
INPUT_DIR = LOCAL_ROOT / 'inputs'
WORKSPACE = LOCAL_ROOT / 'work'
STORE = Path('/content/drive/MyDrive/REWIND-like prototype/reports/indexes/fleet_index_v1/fleet_survey_001')
PREPARED_DIR = STORE.with_name(STORE.name + '_prepared')
OWNER = 'fleet_survey_001'
FAILED = None
BOOTSTRAP_OK = False

In [ ]:
def require_source_modules(source_dir):
    source_dir=source_dir.resolve()
    for name,module in list(sys.modules.items()):
        if any(name==prefix or name.startswith(prefix+'.') for prefix in ('rewind_fleet','rewind_archive','rewind_archive_index')):
            filename=getattr(module,'__file__',None)
            if filename is None or not Path(filename).resolve().is_relative_to(source_dir):
                raise ValueError('fresh_runtime_required_loaded_project_module_from_other_source')
try:
    if ACTION not in ('prepare_and_start','prepare','status','stop','resume','recover'): raise ValueError('invalid_action')
    if len(SOURCE_ZIP_SHA256)!=64 or len(INPUT_ZIP_SHA256)!=64: raise ValueError('root_package_pins_required')
    LOCAL_ROOT.mkdir(parents=True,exist_ok=True)
    from google.colab import files
    needed = [n for n in (SOURCE_ZIP_NAME, INPUT_ZIP_NAME) if not (LOCAL_ROOT/n).is_file()]
    if needed:
        print('두 작은 ZIP 업로드:',needed)
        uploaded = files.upload()
        if set(uploaded) != set(needed): raise ValueError('exact_package_names_required')
        for n,raw in uploaded.items():
            if len(raw)>16*1024**2: raise ValueError('small_bundle_only')
            (LOCAL_ROOT/n).write_bytes(raw)
    def bootstrap_source(archive,expected,target):
        if archive.stat().st_size>16*1024**2 or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected: raise ValueError('source_zip_sha')
        with zipfile.ZipFile(archive) as z:
            infos=z.infolist()
            if len(infos)>4096 or sum(i.file_size for i in infos)>64*1024**2: raise ValueError('bundle_expansion')
            names={}
            for i in infos:
                n=i.filename.rstrip('/')
                if not n or chr(92) in n or ':' in n or n.startswith('/') or any(x in ('','.','..') for x in n.split('/')) or (i.external_attr>>16)&0o170000==0o120000: raise ValueError('bundle_path')
                if i.is_dir(): continue
                if n.casefold() in names: raise ValueError('bundle_duplicate')
                names[n.casefold()]=(n,i)
            if target.exists():
                actual={p.relative_to(target).as_posix().casefold():p for p in target.rglob('*') if p.is_file()}
                if set(actual)!=set(names): raise ValueError('source_reuse_exact_required')
                for key,(n,i) in names.items():
                    if actual[key].is_symlink() or actual[key].read_bytes()!=z.read(i): raise ValueError('source_reuse_hash')
            else:
                target.mkdir(parents=True)
                for n,i in names.values():
                    p=target/n;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(z.read(i))
    bootstrap_source(LOCAL_ROOT/SOURCE_ZIP_NAME,SOURCE_ZIP_SHA256,SOURCE_DIR)
    if str(SOURCE_DIR) not in sys.path: sys.path.insert(0,str(SOURCE_DIR))
    require_source_modules(SOURCE_DIR)
    # package install은 source/media 비용과 분리, 요청 timeout/retry만 적용하고 전체 시간 cap은 없음.
    import importlib.util
    if importlib.util.find_spec('indexed_gzip') is None or importlib.util.find_spec('psutil') is None:
        subprocess.run([sys.executable,'-B','-m','pip','install','--no-cache-dir','--retries','0','--timeout','5','indexed_gzip==1.10.3','psutil>=5.9,<8'],check=True)
    import indexed_gzip,psutil
    if indexed_gzip.__version__!='1.10.3': raise ValueError('indexed_gzip_1_10_3_required_restart_runtime_if_conflicting')
    from rewind_fleet.survey import stage_zip,load_json,prepared,recover
    stage_zip(LOCAL_ROOT/INPUT_ZIP_NAME,INPUT_ZIP_SHA256,INPUT_DIR)
    from rewind_fleet.__main__ import main,authorized_factory
    from rewind_fleet.planning import validate
    POLICY=validate(load_json(INPUT_DIR/'survey_request.json'),'fleet_survey_request_v1')
    if POLICY['content_sha256']!=EXPECTED_REQUEST_SHA256 or POLICY['time_policy']!='observe_only': raise ValueError('approved_request_pin')
    print({'allocation_base':POLICY['approved_base'],'cap':POLICY['approved_cap'],'time_policy':'observe_only','pin_cap':POLICY['pin_cap']})
    BOOTSTRAP_OK=True
except Exception as exc:
    FAILED={'stage':'bootstrap','type':type(exc).__name__,'reason':getattr(exc,'reason',None)}
    print('준비 중단. 마지막 회수 셀을 실행하세요.',FAILED)

In [ ]:
if BOOTSTRAP_OK:
    try:
        from google.colab import auth,drive
        auth.authenticate_user()
        drive.mount('/content/drive')
        WORKSPACE.mkdir(parents=True,exist_ok=True)
        if ACTION in ('prepare_and_start','prepare','resume'):
            if psutil.virtual_memory().available < 2*1024**3 or shutil.disk_usage(LOCAL_ROOT).free < 4*1024**3: raise ValueError('need_2GiB_available_RAM_4GiB_local_disk')
            if ACTION=='resume' and not (PREPARED_DIR/'PREPARED.json').is_file(): raise ValueError('resume_requires_original_persistent_prepared_no_repin')
            OBSERVED=load_json(INPUT_DIR/'observed_inventory.json')
            PREPARED=prepared(OBSERVED,POLICY,PREPARED_DIR,session_factory=lambda:authorized_factory()(),source_sha256=SOURCE_ZIP_SHA256)
            PLAN_SHA256=PREPARED['plan_sha256']
            print({'runtime_inventory_sha256':PREPARED['inventory_sha256'],'expected_plan_sha256':PLAN_SHA256,'prepared':str(PREPARED_DIR),'same_plan_on_resume':True})
        elif (PREPARED_DIR/'PREPARED.json').is_file():
            PREPARED=load_json(PREPARED_DIR/'PREPARED.json');PLAN_SHA256=PREPARED['plan_sha256']
        def survey_status():
            main(['status','--store',str(STORE)])
            if globals().get('WORKER') is not None:
                print({'worker_exit':WORKER.poll(),'wall_since_launch_seconds':time.time()-WORKER_STARTED_AT,'log':str(WORKER_LOG),'report':str(WORKER_REPORT)})
                if WORKER_REPORT.is_file():
                    r=load_json(WORKER_REPORT);print({'actual_worker_wall_seconds':r.get('seconds'),'observed_segment_seconds':r.get('observed_seconds'),'stop_reason':r.get('stop_reason')})
        def survey_stop():
            if not globals().get('PLAN_SHA256'): raise ValueError('prepared_plan_required')
            main(['stop','--store',str(STORE),'--plan-sha256',PLAN_SHA256])
        if ACTION in ('prepare_and_start','resume'):
            if globals().get('WORKER') is not None and WORKER.poll() is None: raise ValueError('worker_already_running')
            job=uuid.uuid4().hex;WORKER_LOG=WORKSPACE/('worker-'+job+'.log');WORKER_REPORT=WORKSPACE/('run-'+job+'.json')
            argv=[sys.executable,'-B','-u','-m','rewind_fleet','resume' if ACTION=='resume' else 'auto-run',
                  '--inventory',str(PREPARED_DIR/'runtime_inventory.json'),'--plan',str(PREPARED_DIR/'plan.json'),'--plan-sha256',PLAN_SHA256,
                  '--store',str(STORE),'--workspace',str(WORKSPACE),'--owner',OWNER,'--single-writer','--session-seconds','none','--output',str(WORKER_REPORT)]
            env=dict(os.environ,PYTHONDONTWRITEBYTECODE='1')
            with WORKER_LOG.open('xb') as log:
                WORKER=subprocess.Popen(argv,cwd=SOURCE_DIR,env=env,stdout=log,stderr=subprocess.STDOUT)
            WORKER_STARTED_AT=time.time()
            print({'pid':WORKER.pid,'time_limit':None,'log':str(WORKER_LOG),'report':str(WORKER_REPORT),'stop':'별도 셀에서 survey_stop()','status':'survey_status()'})
            from IPython.display import display,HTML
            display(HTML('<div id="survey-clock">Worker 시작 후 경과: 0초</div><script>(function(){const start=Date.now(); const node=document.currentScript.previousElementSibling; setInterval(()=>{node.textContent="Worker 시작 후 경과: "+Math.floor((Date.now()-start)/1000)+"초 · 종료 확인은 survey_status()";},1000);})();</script>'))
        elif ACTION=='status':survey_status()
        elif ACTION=='stop':survey_stop()
    except Exception as exc:
        FAILED={'stage':'prepare_or_launch','type':type(exc).__name__,'reason':getattr(exc,'reason',None)}
        print('준비/시작 중단. 회수 셀로 결과를 보존하세요.',FAILED)

In [ ]:
# Worker가 실행 중이어도 이 셀의 함수는 즉시 반환합니다.
# 상태: survey_status() / 정상 중단: survey_stop()
# 새 runtime 재개: 같은 ZIP과 Drive 경로를 유지하고 ACTION='resume'으로 Run all.
# 실패/중단/완료 회수: make_recovery(). Drive의 전체 pack store를 ZIP으로 복사하지 않습니다.
from pathlib import Path
import json,uuid,zipfile,hashlib
def make_recovery(download=True):
    root=globals().get('LOCAL_ROOT',Path('/content/fleet_survey_runtime'));root.mkdir(parents=True,exist_ok=True)
    output=root/('fleet_survey_recovery_'+uuid.uuid4().hex+'.zip')
    if globals().get('BOOTSTRAP_OK') and callable(globals().get('recover')):
        result=recover(output,store=globals().get('STORE'),prepared_root=globals().get('PREPARED_DIR'),workspace=globals().get('WORKSPACE'),policy=globals().get('POLICY'))
    else:
        # bootstrap 실패에도 stdlib만으로 작은 진단을 회수합니다. 원시 예외 메시지/자격증명은 제외합니다.
        with zipfile.ZipFile(output,'x',compression=zipfile.ZIP_DEFLATED) as z:z.writestr('bootstrap_failure.json',json.dumps({'failure':globals().get('FAILED'),'source_revision':globals().get('SOURCE_REVISION'),'source_or_pack_data_included':False},ensure_ascii=False))
        result={'path':str(output),'bytes':output.stat().st_size,'sha256':hashlib.sha256(output.read_bytes()).hexdigest()}
    print(result)
    if download:
        from google.colab import files
        files.download(str(output))
    return result
if globals().get('FAILED') or globals().get('ACTION')=='recover':make_recovery()
else:print('Worker 시작 후 마지막 결과 회수는 make_recovery()를 실행하세요.')